In [ ]:
import os
import numpy as np
import scanpy as sc
import pandas as pd
import cell2location as c2l
from cell2location.utils import select_slide
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from aquarel import load_theme
import cmcrameri

plt.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.linewidth":     0.8,
    "xtick.major.size":   3,
    "ytick.major.size":   3,
    "xtick.labelsize":    7,
    "ytick.labelsize":    7,
    "axes.titlesize":     8,
    "axes.titleweight":   "bold",
    "axes.labelsize":     7,
    "figure.dpi":         300,
})

In [ ]:
DATA_DIR    = "data"
OUT_DIR     = "spot-deconvolution_zheng"

os.makedirs(OUT_DIR, exist_ok=True)
sc.settings.figdir = OUT_DIR

results_folder = OUT_DIR
ref_run_name   = f"{results_folder}/reference_signatures"
run_name       = f"{results_folder}/cell2location_map"

In [ ]:
adata_vis = sc.read_h5ad(os.path.join(run_name, 'spatial.h5ad'))

In [ ]:
adata_vis.obs['class'] = adata_vis.obs['class'].astype(str)
adata_vis.obs['class'] = adata_vis.obs['class'].replace({'Ovary': 'Adnexa'})
adata_vis.obs['PFI'] = adata_vis.obs.PFI.astype(str)
adata_vis.obs['PFI_short_long'] = adata_vis.obs['PFI'].replace({'short': 'short', 'medium': 'long', 'long': 'long'})
adata_vis.obs['patient'] = adata_vis.obs['patient'].astype(str)
adata_vis.obs['histology'] = adata_vis.obs.histology.astype(str)
adata_vis.obs['histology'] = adata_vis.obs['histology'].replace({'Tumor Epithelium': 'Tumor epithelium', 'Mixed TumEpi+Other': 'Mixed', 'Other': 'Stroma'})
adata_vis.obs['anno'] = adata_vis.obs['class'] + '_' + adata_vis.obs['patient'] + '_PFI-' + adata_vis.obs['PFI']

In [ ]:
import os
import squidpy as sq

mpl.rcdefaults()
# add 5% quantile, representing confident cell abundance, 'at least this amount is present',
# to adata.obs with nice names for plotting
adata_vis.obs[adata_vis.uns['mod']['factor_names']] = adata_vis.obsm['q05_cell_abundance_w_sf']

# select one slide
slide = select_slide(adata_vis, 'Paxgene1')
celltypes = adata_vis.uns['mod']['factor_names']

# os.mkdir(f"{run_name}/visualizations/")

for lib in ['Paxgene1', 'Paxgene2', 'Paxgene3', 'Paxgene4']:
    slide = select_slide(adata_vis, lib)
    fig = sc.pl.spatial(slide,
                        cmap='magma',
                        library_id=lib,
                        color=celltypes,
                        ncols=3, size=1.3,
                        img_key=None,
                        frameon=False,
                        # limit color scale at 99.2% quantile of cell abundance
                        vmin=0, vmax='p99.2',
                        return_fig=True,
                       )
    fig.savefig(f"{run_name}/visualizations/{lib}_spatial.pdf", bbox_inches='tight')
    plt.show()

In [ ]:
# ── Assign dominant cell type per spot ────────────────────────────────────
abundance = adata_vis.obsm['means_cell_abundance_w_sf'].copy()
abundance.columns = adata_vis.uns['mod']['factor_names']

# Dominant cell type = column with highest predicted abundance
adata_vis.obs['cell_type'] = abundance.idxmax(axis=1).values

In [ ]:
coords = pd.DataFrame(slide.obsm['spatial'], index=slide.obs_names, columns=['x', 'y'])

In [ ]:
print(coords.describe())

In [ ]:
celltypes

In [ ]:
CELL_TYPE_PALETTE = {
    # ── Lymphoid ───────────────────────────────────────────────
    "B cells":                "#7BA7D4",   # muted blue
    "CD4+ T":                 "#4A90C4",   # medium blue
    "CD8+ T cells":           "#1D5FA6",   # deep blue
    "Natural killer cells":   "#6B4FA0",   # purple (cytotoxic, close to CD8)

    # ── Myeloid ────────────────────────────────────────────────
    "Macrophages":            "#4BAD8E",   # teal green

    # ── Stroma ─────────────────────────────────────────────────
    "Fibroblast":             "#D4A45A",   # warm ochre
    "Other stromal cells":    "#B8834A",   # darker ochre
    "Endothelial cells":      "#C97D4E",   # terracotta

    # ── Tumour ─────────────────────────────────────────────────
    "Cancer epithelial cells":"#C0392B",   # red
    "Proliferative cells":    "#E8A020",   # amber (agnostic lineage)
}

In [ ]:
from cell2location.plt import plot_spatial

clust_labels = celltypes[:5]
clust_col = ['' + str(i) for i in clust_labels]

with mpl.rc_context({'figure.figsize': (15, 15)}):
    for s in ['Paxgene1', 'Paxgene2', 'Paxgene3', 'Paxgene4']:
        slide = select_slide(adata_vis, s)
        plt.figure(figsize=(15, 15))
        plot_spatial(
            adata=slide,
            # labels to show on a plot
            color=clust_col, labels=clust_labels,
            show_img=True,
            # 'fast' (white background) or 'dark_background'
            style='fast',
            # limit color scale at 99.2% quantile of cell abundance
            max_color_quantile=0.992,
            # size of locations (adjust depending on figure size)
            circle_diameter=6,
            colorbar_position='right',
            image_cmap="cmc.lipari"
        )

        plt.savefig(f"{run_name}/visualizations/{s}_spatial-combined.pdf", bbox_inches='tight')
    plt.show()

In [ ]:
clust_labels = celltypes[5:]
clust_col = ['' + str(i) for i in clust_labels]

with mpl.rc_context({'figure.figsize': (15, 15)}):
    for s in ['Paxgene1', 'Paxgene2', 'Paxgene3', 'Paxgene4']:
        slide = select_slide(adata_vis, s)
        plt.figure(figsize=(15, 15))
        plot_spatial(
            adata=slide,
            # labels to show on a plot
            color=clust_col, labels=clust_labels,
            show_img=True,
            # 'fast' (white background) or 'dark_background'
            style='fast',
            # limit color scale at 99.2% quantile of cell abundance
            max_color_quantile=0.992,
            # size of locations (adjust depending on figure size)
            circle_diameter=6,
            colorbar_position='right',
            image_cmap="cmc.lipari"
        )

        plt.savefig(f"{run_name}/visualizations/{s}_spatial-combined2.pdf", bbox_inches='tight')
    plt.show()

### Summarize results per outcome group

In [ ]:
adata_vis = adata_vis[~adata_vis.obs['class'].isin(['Marker'])].copy()

In [ ]:
theme = (
    load_theme("umbra_light")
    .set_grid(draw=False)
    # .set_color(palette='cmc.glasgow')
)

In [ ]:
abundance_key = "q05_cell_abundance_w_sf"
cell_abund = pd.DataFrame(
    adata_vis.obsm[abundance_key],
    index=adata_vis.obs_names
)

if cell_abund.shape[1] == len(getattr(adata_vis.uns.get("mod", {}), "factor_names", [])):
    cell_abund.columns = adata_vis.uns["mod"]["factor_names"]

if cell_abund.columns.dtype == "int64":
    print("Warning: abundance columns are unnamed. You may need to set cell type names manually.")

group_col = "histology"

if group_col not in adata_vis.obs.columns:
    raise KeyError(
        f"'{group_col}' not found in adata.obs. "
        f"Available columns are: {list(adata_vis.obs.columns)}"
    )

cell_abund[group_col] = adata_vis.obs[group_col].values

# Aggregate by tissue type
grouped_sum = cell_abund.groupby(group_col, observed=True).sum()

# Convert to proportions within each tissue type
grouped_prop = grouped_sum.div(grouped_sum.sum(axis=1), axis=0)

grouped_prop.head()

In [ ]:
plt.figure(figsize=(9, 4))
ax = sns.heatmap(
    grouped_prop,
    cmap='cmc.lipari',
    annot=True, fmt=".2f",
    cbar_kws={'label': 'Proportion'}
)
plt.title("Predicted cell-type composition per tissue type", fontweight='bold')

# Rotate both axes
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0, va='center')

plt.xlabel("Cell type")
plt.ylabel("Tissue type")

plt.tight_layout()
plt.savefig(f"{run_name}/visualizations/heatmap_proportions_tissuetype.pdf", bbox_inches='tight')
plt.show()

In [ ]:
celltypes

In [ ]:
clust_labels = ['Macrophages', 'Fibroblast', 'Other stromal cells', 'Endothelial cells', 'Cancer epithelial cells']
clust_col = ['' + str(i) for i in clust_labels]


with mpl.rc_context({'figure.figsize': (15, 15)}):
    for s in ['Paxgene1', 'Paxgene2', 'Paxgene3', 'Paxgene4']:
        slide = select_slide(adata_vis, s)
        fig = plt.figure(figsize=(15, 15))
        plot_spatial(
            adata=slide,
            # labels to show on a plot
            color=clust_col, labels=clust_labels,
            show_img=True,
            # 'fast' (white background) or 'dark_background'
            style='fast',
            # limit color scale at 99.2% quantile of cell abundance
            max_color_quantile=0.992,
            # size of locations (adjust depending on figure size)
            circle_diameter=6,
            colorbar_position='right',
            image_cmap="cmc.lipari"
        )

        fig.savefig(f"{run_name}/visualizations/{s}_spatial-combined_interesting.pdf", bbox_inches='tight')
    plt.show()

In [ ]:
# Compute grouped proportions by both histology and outcome
cell_abundances_df = adata_vis.obsm['means_cell_abundance_w_sf'].copy()
cell_abundances_df.columns = adata_vis.uns['mod']['factor_names']
cell_abundances_df['histology'] = adata_vis.obs['histology'].values
cell_abundances_df['outcome'] = adata_vis.obs['PFI'].values

grouped = cell_abundances_df.groupby(['histology', 'outcome']).mean()
grouped_prop = grouped.div(grouped.sum(axis=1), axis=0)

# Prepare for plotting
plot_df = grouped_prop.reset_index().melt(
    id_vars=['histology', 'outcome'],
    var_name='cell_type',
    value_name='proportion'
)

# Unique outcomes for faceting
outcomes = plot_df['outcome'].unique()

n_outcomes = len(outcomes)
fig, axes = plt.subplots(1, n_outcomes, figsize=(6 * n_outcomes, 3), sharey=True)

if n_outcomes == 1:
    axes = [axes]

for ax, outcome in zip(axes, outcomes):
    pivot = (plot_df[plot_df['outcome'] == outcome]
             .pivot(index='histology', columns='cell_type', values='proportion'))

    sns.heatmap(
        pivot,
        cmap='cmc.lipari',
        cbar_kws={'label': 'Proportion'},
        ax=ax,
        annot=True, fmt=".2f",
    )

    ax.set_title(f"PFI: {outcome}", pad=12, fontweight='bold')
    ax.set_xlabel("Cell type")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

    ax.set_ylabel("Tissue type")
    ax.set_yticklabels(pivot.index, rotation=0, va='center')

plt.tight_layout()
plt.savefig(f"{run_name}/visualizations/heatmap_proportions_tissuetype_by_outcome.pdf", bbox_inches='tight')
plt.show()

In [ ]:
pivot = grouped_prop.copy()
pivot.index = [f"{idx[0]}\n({idx[1]})" for idx in pivot.index]

plt.figure(figsize=(10, 6))
ax = sns.heatmap(pivot, cmap='cmc.lipari', annot=True, fmt=".2f",
                 cbar_kws={'label': 'Proportion'})
plt.title("Predicted cell-type composition per tissue type and outcome")
plt.xlabel("Cell type")
plt.ylabel("Tissue (Outcome)")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{run_name}/visualizations/heatmap_tissue_outcome_combined.pdf", bbox_inches='tight')
plt.show()

In [ ]:
pivot

In [ ]:
cell_abundances_df = adata_vis.obsm['means_cell_abundance_w_sf'].copy()
cell_abundances_df.columns = adata_vis.uns['mod']['factor_names']

cell_abundances_df['site'] = adata_vis.obs['class'].values
cell_abundances_df['histology'] = adata_vis.obs['histology'].values
cell_abundances_df['outcome'] = adata_vis.obs['PFI'].values

grouped = cell_abundances_df.groupby(['site', 'histology', 'outcome']).mean()
grouped_prop = grouped.div(grouped.sum(axis=1), axis=0)

grouped_flat = grouped_prop.reset_index()

grouped_flat['site_str'] = grouped_flat['site'].astype(str)
grouped_flat['histology_str'] = grouped_flat['histology'].astype(str)
grouped_flat['outcome_str'] = grouped_flat['outcome'].astype(str)

grouped_flat['row_label'] = (
    grouped_flat['site_str'] + ' | ' +
    grouped_flat['histology_str'] + ' | ' +
    grouped_flat['outcome_str']
)

grouped_flat = grouped_flat.sort_values(['site_str', 'histology_str', 'outcome_str'])

heatmap_mat = (
    grouped_flat
    .set_index('row_label')
    .drop(columns=['site', 'histology', 'outcome',
                   'site_str', 'histology_str', 'outcome_str'])
)

sites = grouped_flat['site_str']
hists = grouped_flat['histology_str']
outs  = grouped_flat['outcome_str']
row_index = grouped_flat['row_label']

site_palette = dict(zip(
    sites.unique(),
    sns.color_palette("mako", n_colors=sites.nunique())
))

hist_palette = dict(zip(
    hists.unique(),
    sns.color_palette("cubehelix", n_colors=hists.nunique())
))

out_palette = {
    'short': '#DF7B26',
    'medium': '#1A71B8',
    'long': '#51AFA9'
}

row_colors = pd.DataFrame({
    'Site': sites.map(site_palette).values,
    'Histology': hists.map(hist_palette).values,
    'Outcome': outs.map(out_palette).values,
}, index=row_index)

g = sns.clustermap(
    heatmap_mat,
    # cmap=cmc.lipari,
    row_cluster=False,
    col_cluster=False,
    row_colors=row_colors,
    figsize=(12, 8),
    cbar_kws={'label': 'Proportion'}
)

g.ax_heatmap.set_xlabel("Cell type")
g.ax_heatmap.set_ylabel("")

# row labels are already informative: site | histology | outcome
g.ax_heatmap.set_yticklabels(
    g.ax_heatmap.get_yticklabels(),
    rotation=0
)

g.ax_heatmap.set_xticklabels(
    g.ax_heatmap.get_xticklabels(),
    rotation=45,
    ha='right'
)

plt.tight_layout()
plt.savefig(f"{run_name}/visualizations/heatmap_site_histology_outcome_flatindex.pdf",
            bbox_inches='tight')
plt.show()

In [ ]:
cell_abundances_df = adata_vis.obsm['means_cell_abundance_w_sf'].copy()
cell_type_names = adata_vis.uns['mod']['factor_names']
cell_abundances_df.columns = cell_type_names
cell_abundances_df['histology'] = adata_vis.obs['histology'].values

# Group by histology and compute mean cell type abundance
grouped = cell_abundances_df.groupby('histology').mean()

grouped_prop = grouped.div(grouped.sum(axis=1), axis=0)

plot_df = grouped_prop.reset_index().melt(id_vars='histology',
                                          var_name='cell_type',
                                          value_name='proportion')

plt.figure(figsize=(5, 3))
sns.barplot(data=plot_df, x='histology', y='proportion', hue='cell_type', palette="cmc.glasgow")
plt.title('Proportion of Predicted Cell Types per Histological Tissue Type')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.savefig(f'{run_name}/visualizations/proportions_tissuetype.pdf', bbox_inches='tight')
plt.show()

In [ ]:
cell_abundances_df = adata_vis.obsm['means_cell_abundance_w_sf'].copy()
cell_type_names = adata_vis.uns['mod']['factor_names']
cell_abundances_df.columns = cell_type_names
cell_abundances_df['histology'] = adata_vis.obs['histology'].values

# Group by histology and compute mean cell type abundance
grouped = cell_abundances_df.groupby('histology').mean()

grouped_prop = grouped.div(grouped.sum(axis=1), axis=0)

plot_df = grouped_prop.reset_index().melt(id_vars='histology',
                                          var_name='cell_type',
                                          value_name='proportion')

stacked_df = plot_df.pivot(index='histology', columns='cell_type', values='proportion')

# ── Figure ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(3.5, 3))

bottom = pd.Series(0.0, index=stacked_df.index)
for cell_type in stacked_df.columns:
    ax.bar(
        stacked_df.index,
        stacked_df[cell_type],
        bottom=bottom,
        label=cell_type,
        color=CELL_TYPE_PALETTE[cell_type],
        width=0.6,
        linewidth=0,
    )
    bottom += stacked_df[cell_type]

# Grid behind bars
ax.set_axisbelow(True)
ax.yaxis.grid(True, linestyle=":", linewidth=0.5, color="#cccccc")
ax.xaxis.grid(False)

# Y-axis as percentage, fixed 0–100%
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_ylim(0, 1)

# Labels & title
ax.set_title('Cell type proportions\nper tissue compartment', pad=6)
ax.set_xlabel('Tissue compartment')
ax.set_ylabel('Proportion')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Spine weight consistency
for spine in ax.spines.values():
    spine.set_linewidth(0.8)

# Legend outside, no frame
ax.legend(
    title='Cell type',
    title_fontsize=7,
    fontsize=6,
    bbox_to_anchor=(1.02, .5),
    loc='center left',
    borderaxespad=0.,
    frameon=False,
)

fig.savefig(
    f'{run_name}/visualizations/proportions_stacked_histology.pdf',
    bbox_inches='tight',
    dpi=300,
)
plt.show()

In [ ]:
cell_abundances_df = adata_vis.obsm['means_cell_abundance_w_sf'].copy()
cell_type_names = adata_vis.uns['mod']['factor_names']
cell_abundances_df.columns = cell_type_names

cell_abundances_df['PFI'] = adata_vis.obs['PFI'].values

# Group by outcome and compute mean cell type abundance
grouped = cell_abundances_df.groupby('PFI').mean()

grouped_prop = grouped.div(grouped.sum(axis=1), axis=0)

plot_df = grouped_prop.reset_index().melt(id_vars='PFI',
                                          var_name='cell_type',
                                          value_name='proportion')

plt.figure(figsize=(6, 3))
sns.barplot(data=plot_df, x='PFI', y='proportion', hue='cell_type', palette="muted")
plt.title('Proportion of predicted cell types per outcome group')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Progression-free interval')
plt.ylabel('Proportion')
plt.tight_layout()

ax.set_axisbelow(True)
ax.yaxis.grid(True, linestyle=":", linewidth=0.5, color="#cccccc")
ax.xaxis.grid(False)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.savefig(f'{run_name}/visualizations/proportions_PFI.pdf', bbox_inches='tight')
plt.show()

In [ ]:
cell_abundances_df = adata_vis.obsm['means_cell_abundance_w_sf'].copy()
cell_type_names = adata_vis.uns['mod']['factor_names']
cell_abundances_df.columns = cell_type_names

cell_abundances_df['PFI'] = adata_vis.obs['PFI'].values

# Group by outcome and compute mean cell type abundance
grouped = cell_abundances_df.groupby('PFI').mean()

grouped_prop = grouped.div(grouped.sum(axis=1), axis=0)

plot_df = grouped_prop.reset_index().melt(id_vars='PFI',
                                          var_name='cell_type',
                                          value_name='proportion')

stacked_df = plot_df.pivot(index='PFI', columns='cell_type', values='proportion')
fig, ax = plt.subplots(figsize=(3.5, 3))

bottom = pd.Series(0.0, index=stacked_df.index)
for i, cell_type in enumerate(stacked_df.columns):
    ax.bar(
        stacked_df.index,
        stacked_df[cell_type],
        bottom=bottom,
        label=cell_type,
        color=CELL_TYPE_PALETTE[cell_type],
        width=0.6,
        linewidth=0,
    )
    bottom += stacked_df[cell_type]

# Grid behind bars
ax.set_axisbelow(True)
ax.yaxis.grid(True, linestyle=":", linewidth=0.5, color="#cccccc")
ax.xaxis.grid(False)

# Y-axis as percentage, fixed 0–100 %
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_ylim(0, 1)

# Labels & title
ax.set_title('Cell type proportions\nper outcome group', pad=6)
ax.set_xlabel('Platinum-free interval')
ax.set_ylabel('Proportion')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Spine weight consistency
for spine in ax.spines.values():
    spine.set_linewidth(0.8)

# Legend outside, no frame
ax.legend(
    title='Cell type',
    title_fontsize=7,
    fontsize=6,
    bbox_to_anchor=(1.02, .5),
    loc='center left',
    borderaxespad=0.,
    frameon=False,
)

fig.savefig(
    f'{run_name}/visualizations/proportions_stacked_PFI.pdf',
    bbox_inches='tight',
    dpi=300,
)
plt.show()

In [ ]:
import string

def make_stacked_df(adata, group_col):
    """Compute normalised cell type proportion stacked bar data."""
    df = adata.obsm['means_cell_abundance_w_sf'].copy()
    df.columns = adata.uns['mod']['factor_names']
    df[group_col] = adata.obs[group_col].values
    grouped = df.groupby(group_col).mean()
    grouped_prop = grouped.div(grouped.sum(axis=1), axis=0)
    stacked = grouped_prop.reset_index().melt(
        id_vars=group_col, var_name='cell_type', value_name='proportion'
    ).pivot(index=group_col, columns='cell_type', values='proportion')
    return stacked

def plot_stacked_bar(ax, stacked_df, xlabel, title, ylabel=True):
    """Plot a stacked bar chart on a given axis."""
    bottom = pd.Series(0.0, index=stacked_df.index)
    for cell_type in stacked_df.columns:
        ax.bar(
            stacked_df.index,
            stacked_df[cell_type],
            bottom=bottom,
            label=cell_type,
            color=CELL_TYPE_PALETTE[cell_type],
            width=0.6,
            linewidth=0,
        )
        bottom += stacked_df[cell_type]

    ax.set_axisbelow(True)
    ax.yaxis.grid(True, linestyle=":", linewidth=0.5, color="#cccccc")
    ax.xaxis.grid(False)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.set_ylim(0, 1)
    ax.set_title(title, pad=6)
    ax.set_xlabel(xlabel)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
    if ylabel:
        ax.set_ylabel('Proportion')
    else:
        ax.set_ylabel('')
        ax.set_yticklabels([])

# ── Data ───────────────────────────────────────────────────────────────────
stacked_hist = make_stacked_df(adata_vis, 'histology')
stacked_pfi  = make_stacked_df(adata_vis, 'PFI')
stacked_class  = make_stacked_df(adata_vis, 'class')

# ── Figure ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(
    1, 3,
    figsize=(6, 3),
    sharey=True,
)
fig.set_layout_engine('constrained', w_pad=0.05, wspace=0.05)

plot_stacked_bar(axes[0], stacked_hist,
                 xlabel='Compartment',
                 title='Cell type proportions\nper tissue compartment',
                 ylabel=True)
plot_stacked_bar(axes[1], stacked_pfi,
                 xlabel='Platinum-free interval',
                 title='Cell type proportions\nper outcome group',
                 ylabel=False)
plot_stacked_bar(axes[2], stacked_class,
                 xlabel='Site',
                 title='Cell type proportions\nper tumor site',
                 ylabel=False)

# ── Shared legend to the right ─────────────────────────────────────────
handles = [
    mpatches.Patch(facecolor=CELL_TYPE_PALETTE[ct], label=ct, linewidth=0)
    for ct in stacked_hist.columns
]
_, labels = ax.get_legend_handles_labels()
    
fig.legend(
    handles[::-1], labels[::-1],
    title='Cell type',
    title_fontsize=7,
    fontsize=6,
    bbox_to_anchor=(1.02, 0.6),
    loc='center left',
    borderaxespad=0.,
    frameon=False,
)

fig.savefig(
    f'{run_name}/visualizations/proportions_stacked_combined.png',
    bbox_inches='tight',
    dpi=600,
)
plt.show()

In [ ]:
cell_abundances_df = adata_vis.obsm['means_cell_abundance_w_sf'].copy()
cell_type_names = adata_vis.uns['mod']['factor_names']
cell_abundances_df.columns = cell_type_names
cell_abundances_df['PFI'] = adata_vis.obs['PFI'].values

grouped = cell_abundances_df.groupby('PFI').mean()
grouped_prop = grouped.div(grouped.sum(axis=1), axis=0)
plot_df = grouped_prop.reset_index().melt(
    id_vars='PFI', var_name='cell_type', value_name='proportion'
)

fig, ax = plt.subplots(figsize=(5, 2))

sns.barplot(
    data=plot_df,
    x='PFI',
    y='proportion',
    hue='cell_type',
    palette="muted",
    linewidth=0.4,
    edgecolor="white",
    ax=ax,
)

# Grid behind bars
ax.set_axisbelow(True)
ax.yaxis.grid(True, linestyle=":", linewidth=0.5, color="#cccccc")
ax.xaxis.grid(False)

# Y-axis as percentage
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))

# Labels & title
ax.set_title('Proportion of predicted cell types per outcome group', pad=6)
ax.set_xlabel('Progression-free interval')
ax.set_ylabel('Proportion')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Legend outside the plot
ax.legend(
    title='Cell type',
    title_fontsize=7,
    fontsize=6,
    bbox_to_anchor=(1.02, 1.),
    loc='upper left',
    borderaxespad=0.,
    frameon=False,       # cleaner without box
)

fig.savefig(
    f'{run_name}/visualizations/proportions_PFI.pdf',
    bbox_inches='tight',
    dpi=300,
)
plt.show()

In [ ]:
cell_abundances_df = adata_vis.obsm['means_cell_abundance_w_sf'].copy()
cell_type_names = adata_vis.uns['mod']['factor_names']
cell_abundances_df.columns = cell_type_names

cell_abundances_df['class'] = adata_vis.obs['class'].values

# Group by class and compute mean cell type abundance
grouped = cell_abundances_df.groupby('class').mean()

grouped_prop = grouped.div(grouped.sum(axis=1), axis=0)

plot_df = grouped_prop.reset_index().melt(id_vars='class',
                                          var_name='cell_type',
                                          value_name='proportion')

plt.figure(figsize=(5, 3))
sns.barplot(data=plot_df, x='class', y='proportion', hue='cell_type', palette="cmc.lipari")
plt.title('Proportion of Predicted Cell Types per Site of Origin')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.savefig(f'{run_name}/visualizations/proportions_class.pdf', bbox_inches='tight')
plt.show()

### Identifying distinct regions in the tissue

In [ ]:
# compute KNN using the cell2location output stored in adata.obsm
sc.pp.neighbors(adata_vis, use_rep='q05_cell_abundance_w_sf',
                n_neighbors = 15)

# Cluster spots into regions using scanpy
sc.tl.leiden(adata_vis, resolution=.6, flavor='igraph')

# add region as categorical variable
adata_vis.obs["region_cluster"] = adata_vis.obs["leiden"].astype("category")

In [ ]:
SLIDES = ['Paxgene1', 'Paxgene2', 'Paxgene3', 'Paxgene4']

# ── Helpers ────────────────────────────────────────────────────────────────
def add_umap_style(ax, title):
    """Minimal UMAP axis: no ticks, labelled arrows for axes."""
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(title, pad=6)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    # Small UMAP-style axis arrows
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    xr = (xlim[1] - xlim[0]) * 0.12
    yr = (ylim[1] - ylim[0]) * 0.12
    arrow_kw = dict(arrowstyle='->', color='#444444',
                    lw=0.8, mutation_scale=8)
    ax.annotate('', xy=(xlim[0] + xr, ylim[0]),
                xytext=(xlim[0], ylim[0]),
                arrowprops=arrow_kw, annotation_clip=False)
    ax.annotate('', xy=(xlim[0], ylim[0] + yr),
                xytext=(xlim[0], ylim[0]),
                arrowprops=arrow_kw, annotation_clip=False)
    ax.text(xlim[0] + xr / 2, ylim[0] - (ylim[1]-ylim[0])*0.04,
            'UMAP1', fontsize=6, ha='center', va='top', color='#444444')
    ax.text(xlim[0] - (xlim[1]-xlim[0])*0.03, ylim[0] + yr / 2,
            'UMAP2', fontsize=6, ha='right', va='center',
            color='#444444', rotation=90)

def encode_categorical(labels):
    """Map string labels → integer codes + ordered unique list."""
    uniq = list(dict.fromkeys(labels))       # preserve order, deduplicate
    code = np.array([uniq.index(l) for l in labels])
    return code, uniq

In [ ]:
# ── Reset background colors ────────────────────────────
mpl.rcParams['axes.facecolor']   = 'white'
mpl.rcParams['figure.facecolor'] = 'white'

# ── 1. UMAP — region_cluster ───────────────────────────────────────────────
sc.tl.umap(adata_vis, min_dist=0.3, spread=1)
umap_xy = adata_vis.obsm['X_umap']

region_labels = adata_vis.obs['region_cluster'].astype(str).values
region_codes, region_uniq = encode_categorical(region_labels)
n_regions = len(region_uniq)
region_palette = plt.get_cmap('RdPu')(np.linspace(0.25, 0.95, n_regions))

fig, ax = plt.subplots(figsize=(2, 2))
for i, region in enumerate(region_uniq):
    mask = region_codes == i
    ax.scatter(umap_xy[mask, 0], umap_xy[mask, 1],
               c=[region_palette[i]], s=5, linewidths=0,
               alpha=0.8, rasterized=True, label=region)
    # Centroid label
    cx, cy = umap_xy[mask, 0].mean(), umap_xy[mask, 1].mean()
    ax.text(cx, cy, region, fontsize=6, ha='center', va='center',
            fontweight='bold', color='white',
            bbox=dict(boxstyle='round,pad=0.15', fc=region_palette[i],
                      ec='none', alpha=0.75))
add_umap_style(ax, 'Region cluster')
fig.savefig(f'{run_name}/visualizations/umap_region_cluster.png',
            bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
# ── 2. UMAP — sample ──────────────────────────────────────────────────────
sample_labels = adata_vis.obs['sample'].astype(str).values
sample_codes, sample_uniq = encode_categorical(sample_labels)
n_samples = len(sample_uniq)
sample_palette = plt.get_cmap('RdPu')(np.linspace(0.25, 0.95, n_samples))

fig, ax = plt.subplots(figsize=(2, 2))
for i, sample in enumerate(sample_uniq):
    mask = sample_codes == i
    ax.scatter(umap_xy[mask, 0], umap_xy[mask, 1],
               c=[sample_palette[i]], s=5, linewidths=0,
               alpha=0.8, rasterized=True, label=sample)
add_umap_style(ax, 'Sample')
ax.legend(
    title='TMA', title_fontsize=7, fontsize=6,
    markerscale=1.5, frameon=False,
    bbox_to_anchor=(1.02, .5), loc='center left', borderaxespad=0.,
)
fig.savefig(f'{run_name}/visualizations/umap_sample.png',
            bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
# ── 2. UMAP — patient ──────────────────────────────────────────────────────
sample_labels = adata_vis.obs['patient'].astype(str).values
sample_codes, sample_uniq = encode_categorical(sample_labels)
n_samples = len(sample_uniq)
sample_palette = plt.get_cmap('RdPu')(np.linspace(0.25, 0.95, n_samples))

fig, ax = plt.subplots(figsize=(2, 2))
for i, sample in enumerate(sample_uniq):
    mask = sample_codes == i
    ax.scatter(umap_xy[mask, 0], umap_xy[mask, 1],
               c=[sample_palette[i]], s=5, linewidths=0,
               alpha=0.8, rasterized=True, label=sample)
add_umap_style(ax, 'Patient')
ax.legend(
    title='Patient', title_fontsize=7, fontsize=6,
    markerscale=1.5, frameon=False,
    bbox_to_anchor=(1.02, .5), loc='center left', borderaxespad=0.,
)
fig.savefig(f'{run_name}/visualizations/umap_patient.png',
            bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
# ── UMAP — cell type ───────────────────────────────────────────────────────
cell_labels = adata_vis.obs['cell_type'].astype(str).values
cell_codes, cell_uniq = encode_categorical(cell_labels)

fig, ax = plt.subplots(figsize=(2, 2))
for i, ct in enumerate(cell_uniq):
    mask = cell_codes == i
    ax.scatter(umap_xy[mask, 0], umap_xy[mask, 1],
               c=[CELL_TYPE_PALETTE.get(ct, '#999999')],
               s=5, linewidths=0,
               alpha=0.8, rasterized=True, label=ct)
add_umap_style(ax, 'Cell type')
ax.legend(
    title='Cell type', title_fontsize=7, fontsize=6,
    markerscale=1.5, frameon=False,
    bbox_to_anchor=(1.02, .5), loc='center left', borderaxespad=0.,
)
fig.savefig(f'{run_name}/visualizations/umap_cell_type.png',
            bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
# ── UMAP — PFI ─────────────────────────────────────────────────────────────
pfi_labels = adata_vis.obs['PFI'].astype(str).values
pfi_codes, pfi_uniq = encode_categorical(pfi_labels)
n_pfi = len(pfi_uniq)

# Neutral categorical palette — PFI has no biological colour convention
pfi_palette = sns.color_palette("colorblind", n_colors=n_pfi)

fig, ax = plt.subplots(figsize=(2, 2))
for i, pfi in enumerate(pfi_uniq):
    mask = pfi_codes == i
    ax.scatter(umap_xy[mask, 0], umap_xy[mask, 1],
               c=[pfi_palette[i]], s=5, linewidths=0,
               alpha=0.8, rasterized=True, label=pfi)
add_umap_style(ax, 'Platinum-free interval')
ax.legend(
    title='PFI', title_fontsize=7, fontsize=6,
    markerscale=1.5, frameon=False,
    bbox_to_anchor=(1.02, .5), loc='center left', borderaxespad=0.,
)
fig.savefig(f'{run_name}/visualizations/umap_PFI.png',
            bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
# ── 3. Spatial — region_cluster per slide ─────────────────────────────────
fig, axes = plt.subplots(1, len(SLIDES),
                         figsize=(4 * len(SLIDES), 4.5),
                         constrained_layout=True)

for ax, s in zip(axes, SLIDES):
    slide      = select_slide(adata_vis, s)
    coords     = slide.obsm['spatial']
    scalef     = slide.uns['spatial'][s]['scalefactors']['tissue_hires_scalef']
    xy         = coords * scalef

    r_labels   = slide.obs['region_cluster'].astype(str).values
    r_codes, _ = encode_categorical(r_labels)   # use global region_uniq
    # Re-map against global order so colours are consistent across slides
    r_codes    = np.array([region_uniq.index(l)
                           if l in region_uniq else -1
                           for l in r_labels])
    colors_pts = [region_palette[c] if c >= 0 else (0.5, 0.5, 0.5, 1.)
                  for c in r_codes]

    ax.scatter(xy[:, 0], xy[:, 1],
               c=colors_pts, s=10, linewidths=0,
               alpha=0.5, rasterized=True)
    ax.set_title(s, pad=4, color='white')
    ax.set_facecolor('black')
    ax.set_axis_off()

# Shared legend
handles = [
    mpatches.Patch(facecolor=region_palette[i], label=r, linewidth=0)
    for i, r in enumerate(region_uniq)
]
fig.legend(handles=handles, title='Region cluster',
           title_fontsize=7, fontsize=6,
           loc='lower center', ncol=n_regions,
           frameon=False, bbox_to_anchor=(0.5, -0.04),
           labelcolor='black')

fig.savefig(f'{run_name}/visualizations/spatial_region_cluster.pdf',
            bbox_inches='tight', dpi=300)
plt.show()

### Co-location of cell types

In [ ]:
# ── 1. Compute colocation scores ───────────────────────────────────────────
# Get normalised abundance proportions per spot
abundance = adata_vis.obsm['means_cell_abundance_w_sf'].copy()
abundance.columns = adata_vis.uns['mod']['factor_names']
props = abundance.div(abundance.sum(axis=1), axis=0)

# Pearson correlation between cell type proportions across spots
# High correlation = cell types tend to co-occur in the same spots
coloc = props.corr(method='pearson')

# ── 2. Cluster cell types by colocation pattern ────────────────────────────
# Store in adata for downstream use
adata_vis.uns['colocation_matrix'] = coloc.values
adata_vis.uns['colocation_names']  = list(coloc.columns)

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import squareform
import matplotlib.colors as mcolors

# ── Clustered heatmap ──────────────────────────────────────────────────
# Hierarchical clustering on correlation distance
dist     = 1 - coloc.values
np.fill_diagonal(dist, 0)
dist     = np.clip(dist, 0, None)          # guard against float imprecision
linkage_mat = linkage(squareform(dist), method='average')
order    = leaves_list(linkage_mat)
coloc_ord = coloc.iloc[order, order]

fig, ax = plt.subplots(figsize=(3.5, 3))

cmap = mcolors.LinearSegmentedColormap.from_list(
    'coloc', ['#2166ac', '#f7f7f7', '#d6604d']
)
im = ax.imshow(coloc_ord.values, cmap=cmap, vmin=-1, vmax=1,
               aspect='auto', interpolation='none', rasterized=True)

ticks = list(coloc_ord.columns)
ax.set_xticks(range(len(ticks)))
ax.set_yticks(range(len(ticks)))
ax.set_xticklabels(ticks, rotation=45, ha='right', fontsize=6)
ax.set_yticklabels([])

strip_w = 0.05
gap = 0.02

strip_ax = ax.inset_axes([-strip_w - gap, 0, strip_w, 1],
                          transform=ax.transAxes)

for idx, ct in enumerate(coloc_ord.index):
    strip_ax.barh(idx, 1,
                  color=CELL_TYPE_PALETTE.get(ct, '#999999'),
                  linewidth=0)
    strip_ax.text(
        -0.05, idx, ct,
        ha='right', va='center', fontsize=6,
    )

strip_ax.set_xlim(0, 1)
strip_ax.set_ylim(-0.5, len(coloc_ord.index) - 0.5)
strip_ax.invert_yaxis()
strip_ax.axis('off')

cb = fig.colorbar(im, ax=ax, shrink=0.5, pad=0.02, aspect=20)
cb.set_label('Pearson r', fontsize=6)
cb.ax.tick_params(labelsize=5)
cb.outline.set_linewidth(0.4)

ax.set_title('Cell type colocation\n(Pearson correlation)', pad=6)

fig.savefig(f'{run_name}/visualizations/colocation_heatmap.pdf',
            bbox_inches='tight')
plt.show()

In [ ]:
# ── Compute per-group colocation matrices ──────────────────────────────────
abundance = adata_vis.obsm['means_cell_abundance_w_sf'].copy()
abundance.columns = adata_vis.uns['mod']['factor_names']
abundance['PFI'] = adata_vis.obs['PFI'].values

pfi_groups = sorted(abundance['PFI'].unique())
coloc_per_pfi = {}

for group in pfi_groups:
    subset = abundance[abundance['PFI'] == group].drop(columns='PFI')
    coloc_per_pfi[group] = subset.corr(method='pearson')

# ── Compute a common clustering order from the global colocation ───────────
# Use the global coloc matrix order so groups are directly comparable
dist = 1 - coloc.values
np.fill_diagonal(dist, 0)
dist = np.clip(dist, 0, None)
linkage_mat = linkage(squareform(dist), method='average')
order = leaves_list(linkage_mat)
cell_type_order = coloc.index[order].tolist()

# ── Plot side by side ──────────────────────────────────────────────────────
n_groups = len(pfi_groups)
cmap = mcolors.LinearSegmentedColormap.from_list(
    'coloc', ['#2166ac', '#f7f7f7', '#d6604d']
)

fig, axes = plt.subplots(1, n_groups,
                          figsize=(9, 2.5))
fig.set_layout_engine('constrained', w_pad=5, wspace=-5)

for ax, group in zip(axes, pfi_groups):
    mat = coloc_per_pfi[group].loc[cell_type_order, cell_type_order]

    im = ax.imshow(mat.values, cmap=cmap, vmin=-1, vmax=1,
                   aspect='auto', interpolation='none', rasterized=True)

    ax.set_xticks(range(len(cell_type_order)))
    ax.set_xticklabels(cell_type_order, rotation=45, ha='right', fontsize=6)
    ax.set_yticks([])
    ax.set_title(f'PFI: {group}', pad=6)

    if ax == axes[0]:
        strip_w = 0.05
        gap = 0.04
        strip_ax = ax.inset_axes([-strip_w - gap, 0, strip_w, 1],
                                   transform=ax.transAxes)
        for idx, ct in enumerate(cell_type_order):
            strip_ax.barh(idx, 1,
                          color=CELL_TYPE_PALETTE.get(ct, '#999999'),
                          linewidth=0)
            strip_ax.text(-0.25, idx, ct, ha='right', va='center', fontsize=6)
        strip_ax.set_xlim(0, 1)
        strip_ax.set_ylim(-0.5, len(cell_type_order) - 0.5)
        strip_ax.invert_yaxis()
        strip_ax.axis('off')

# Shared colorbar on the right
cb = fig.colorbar(im, ax=axes, shrink=0.5, pad=0.02, aspect=20)
cb.set_label('Pearson r', fontsize=6)
cb.ax.tick_params(labelsize=5)
cb.outline.set_linewidth(0.4)

fig.suptitle('Cell type colocation by PFI group', fontsize=14,
             fontweight='bold', y=1.08)

fig.savefig(f'{run_name}/visualizations/colocation_heatmap_PFI.pdf',
            bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# ── Network graph — strong colocation only ─────────────────────────────
import networkx as nx

THRESHOLD = 0.3    # only show edges with |r| > threshold

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_aspect('equal')

G = nx.Graph()
cell_types = list(coloc.columns)
G.add_nodes_from(cell_types)

for i, ct1 in enumerate(cell_types):
    for j, ct2 in enumerate(cell_types):
        if j <= i:
            continue
        r = coloc.loc[ct1, ct2]
        if abs(r) > THRESHOLD:
            G.add_edge(ct1, ct2, weight=r)

pos = nx.spring_layout(G, seed=42, weight='weight', k=1.8)

node_colors = [CELL_TYPE_PALETTE.get(ct, '#999999') for ct in G.nodes]
node_sizes  = [300] * len(G.nodes)

# Positive edges (co-localise) = coral, negative (exclude) = steel blue
pos_edges = [(u, v) for u, v, d in G.edges(data=True) if d['weight'] >  0]
neg_edges = [(u, v) for u, v, d in G.edges(data=True) if d['weight'] <= 0]
pos_widths = [G[u][v]['weight'] * 4  for u, v in pos_edges]
neg_widths = [abs(G[u][v]['weight']) * 4 for u, v in neg_edges]

nx.draw_networkx_edges(G, pos, edgelist=pos_edges, width=pos_widths,
                       edge_color='#d6604d', alpha=0.7, ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=neg_edges, width=neg_widths,
                       edge_color='#2166ac', alpha=0.7, ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                       node_size=node_sizes, linewidths=0.5,
                       edgecolors='white', ax=ax)
nx.draw_networkx_labels(G, pos, font_size=5.5,
                        font_color='black', ax=ax)

# Manual legend
handles = [
    mpl.lines.Line2D([0], [0], color='#d6604d', lw=2, label='Co-localise (r > 0)'),
    mpl.lines.Line2D([0], [0], color='#2166ac', lw=2, label='Exclude (r < 0)'),
]
ax.legend(handles=handles, fontsize=6, frameon=False,
          loc='lower left', borderaxespad=0.)
ax.set_title('Cell type colocation network', fontsize=8,
             fontweight='bold', pad=6)
ax.axis('off')

fig.savefig(f'{run_name}/visualizations/colocation_network.pdf',
            bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
from cell2location import run_colocation

mpl.rcParams['figure.constrained_layout.use'] = False
mpl.rcParams['figure.autolayout'] = False

res_dict, adata_vis = run_colocation(
    adata_vis,
    verbose=False,
    return_all=True,
    model_name='CoLocatedGroupsSklearnNMF',
    train_args={
      'n_fact': np.arange(5, 30),
      'sample_name_col': 'sample', # columns in adata_vis.obs that identifies sample
      'n_restarts': 3 # number of training restarts
    },
    # the hyperparameters of NMF can be also adjusted:
    model_kwargs={'alpha': 0.01, 'init': 'random', "nmf_kwd_args": {"tol": 0.000001}},
    export_args={'path': f'{run_name}/CoLocatedComb/'}
)